In [2]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
import time
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_EMBEDDINGS_API_KEY = os.environ.get("OPENROUTER_EMBEDDINGS_API_KEY")
if not OPENROUTER_EMBEDDINGS_API_KEY:
    raise ValueError("OPENROUTER_EMBEDDINGS_API_KEY در فایل .env تنظیم نشده است.")

LAW_ROOT_DIR = r"F:\Thesis\project\2-RAG\raw_laws"
OUTPUT_DIR = r"F:\Thesis\project\2-RAG\vector_store_bge"
MODEL_NAME = "baai/bge-m3"
BATCH_SIZE = 100

def find_tsv_files(root_dir):
    tsv_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".tsv"):
                tsv_files.append(os.path.join(root, file))
    return tsv_files

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_EMBEDDINGS_API_KEY,
)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

files = find_tsv_files(LAW_ROOT_DIR)
print(f"Found {len(files)} files to process.")

for i, file_path in enumerate(files):
    file_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(files)}] Processing: {file_name}")

    base_name = os.path.splitext(file_name)[0]
    output_path = os.path.join(OUTPUT_DIR, f"{base_name}_embeddings.npy")

    if os.path.exists(output_path):
        print("  -> Already exists. Skipping.")
        continue

    try:
        df = pd.read_csv(file_path, sep="\t")
        texts = df["text"].astype(str).tolist()
        print(f"  -> Loaded {len(texts)} records.")

        if not texts:
            print("  -> Warning: Empty file.")
            continue

    except Exception as e:
        print(f"  -> Error reading file: {e}")
        continue

    all_embeddings = []
    print(f"  -> Generating embeddings in batches of {BATCH_SIZE}...")
    try:
        for j in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[j : j + BATCH_SIZE]

            response = client.embeddings.create(
                model=MODEL_NAME,
                input=batch_texts
            )

            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)

            print(f"    -> Batch {j//BATCH_SIZE + 1} done. ({len(all_embeddings)}/{len(texts)})")
            time.sleep(0.5)

        embeddings_np = np.array(all_embeddings)
        if len(embeddings_np) != len(texts):
            print(f"  -> Warning: embeddings count {len(embeddings_np)} != texts {len(texts)}")

        np.save(output_path, embeddings_np)
        print(f"  -> Saved to {os.path.basename(output_path)}")

    except Exception as e:
        print(f"  -> Error during API call: {e}")

print("\nAll done!")

Found 64 files to process.

[1/64] Processing: ahkam-asl-44_articles.tsv
  -> Already exists. Skipping.

[2/64] Processing: ahkam-daemi-tosee-keshvar_articles.tsv
  -> Loaded 73 records.
  -> Generating embeddings in batches of 100...
    -> Batch 1 done. (73/73)
  -> Saved to ahkam-daemi-tosee-keshvar_articles_embeddings.npy

[3/64] Processing: bazar-oraghbahadar_articles.tsv
  -> Loaded 60 records.
  -> Generating embeddings in batches of 100...
    -> Batch 1 done. (60/60)
  -> Saved to bazar-oraghbahadar_articles_embeddings.npy

[4/64] Processing: hemayat-kala-irani_articles.tsv
  -> Loaded 24 records.
  -> Generating embeddings in batches of 100...
    -> Batch 1 done. (24/24)
  -> Saved to hemayat-kala-irani_articles_embeddings.npy

[5/64] Processing: hesabdari_articles.tsv
  -> Loaded 1 records.
  -> Generating embeddings in batches of 100...
    -> Batch 1 done. (1/1)
  -> Saved to hesabdari_articles_embeddings.npy

[6/64] Processing: sabt-sherkat_articles.tsv
  -> Loaded 12 re

In [3]:
# ===========================
# پردازش سه فایل آرای خاص (که ستون vote_text دارند)
# ===========================

special_files = [
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\dadnameh_metadata.tsv",
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_divan-ali_metadata.tsv",
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_edalat-edari_metadata.tsv"
]

print(f"\nProcessing {len(special_files)} special vote files...")

for i, file_path in enumerate(special_files):
    file_name = os.path.basename(file_path)
    print(f"\n[Special {i+1}/{len(special_files)}] Processing: {file_name}")
    
    base_name = os.path.splitext(file_name)[0]
    output_path = os.path.join(OUTPUT_DIR, f"{base_name}_embeddings.npy")
    
    if os.path.exists(output_path):
        print("  -> Already exists. Skipping.")
        continue
    
    try:
        df = pd.read_csv(file_path, sep="\t")
        
        # استفاده از ستون vote_text به جای text
        if "vote_text" not in df.columns:
            print(f"  -> Error: 'vote_text' column not found.")
            continue
            
        texts = df["vote_text"].astype(str).tolist()
        print(f"  -> Loaded {len(texts)} records.")
        
        if not texts:
            print("  -> Warning: Empty file.")
            continue
            
    except Exception as e:
        print(f"  -> Error reading file: {e}")
        continue
    
    all_embeddings = []
    print(f"  -> Generating embeddings in batches of {BATCH_SIZE}...")
    try:
        for j in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[j : j + BATCH_SIZE]
            
            response = client.embeddings.create(
                model=MODEL_NAME,
                input=batch_texts
            )
            
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
            
            print(f"    -> Batch {j//BATCH_SIZE + 1} done. ({len(all_embeddings)}/{len(texts)})")
            time.sleep(0.5)
            
        embeddings_np = np.array(all_embeddings)
        
        if len(embeddings_np) != len(texts):
            print(f"  -> Warning: embeddings count {len(embeddings_np)} != texts {len(texts)}")
        
        np.save(output_path, embeddings_np)
        print(f"  -> Saved to {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  -> Error during API call: {e}")

print("\nSpecial files processing complete!")


Processing 3 special vote files...

[Special 1/3] Processing: dadnameh_metadata.tsv
  -> Loaded 27914 records.
  -> Generating embeddings in batches of 100...
    -> Batch 1 done. (100/27914)
    -> Batch 2 done. (200/27914)
    -> Batch 3 done. (300/27914)
    -> Batch 4 done. (400/27914)
    -> Batch 5 done. (500/27914)
    -> Batch 6 done. (600/27914)
    -> Batch 7 done. (700/27914)
    -> Batch 8 done. (800/27914)
    -> Batch 9 done. (900/27914)
    -> Batch 10 done. (1000/27914)
    -> Batch 11 done. (1100/27914)
    -> Batch 12 done. (1200/27914)
    -> Batch 13 done. (1300/27914)
    -> Batch 14 done. (1400/27914)
    -> Batch 15 done. (1500/27914)
    -> Batch 16 done. (1600/27914)
    -> Batch 17 done. (1700/27914)
    -> Batch 18 done. (1800/27914)
    -> Batch 19 done. (1900/27914)
    -> Batch 20 done. (2000/27914)
    -> Batch 21 done. (2100/27914)
    -> Batch 22 done. (2200/27914)
    -> Batch 23 done. (2300/27914)
    -> Batch 24 done. (2400/27914)
    -> Batch 25 do